# Ch.01-04 인터랙티브 4D · 5D feature

| 차원 | 시각화 | k-NN |
|------|--------|------|
| **4D** | scatter_matrix + 6 panel pair plot | 4 feature 거리 |
| **5D** | parallel coordinates | 5 feature 거리 |

> 4D·5D **공간**을 한 장에 그릴 수는 없지만, **거리 계산**은 동일합니다.


In [ ]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


def make_fish_3d(n_per_class=25):
    domi = pd.DataFrame({
        "length": rng.normal(35, 3, n_per_class),
        "weight": rng.normal(700, 80, n_per_class),
        "fin": rng.normal(12, 1.5, n_per_class),
        "species": "domi",
    })
    bream = pd.DataFrame({
        "length": rng.normal(28, 2, n_per_class),
        "weight": rng.normal(180, 30, n_per_class),
        "fin": rng.normal(8, 1.2, n_per_class),
        "species": "bream",
    })
    return pd.concat([domi, bream], ignore_index=True)


def make_fish_4d(n_per_class=25):
    df = make_fish_3d(n_per_class)
    df["age"] = np.where(
        df["species"] == "domi",
        rng.normal(3, 1, len(df)),
        rng.normal(2, 0.8, len(df)),
    )
    return df


def make_fish_5d(n_per_class=25):
    df = make_fish_4d(n_per_class)
    df["brightness"] = np.where(
        df["species"] == "domi",
        rng.normal(0.7, 0.1, len(df)),
        rng.normal(0.4, 0.1, len(df)),
    )
    return df

df4 = make_fish_4d()
df5 = make_fish_5d()
df4.head()


## §1 scatter_matrix (4D)


In [ ]:
import plotly.express as px

fig_matrix = px.scatter_matrix(
    df4,
    dimensions=["length", "weight", "fin", "age"],
    color="species",
    color_discrete_map={"domi": "orange", "bream": "steelblue"},
    title="4D: scatter_matrix (pair-wise 2D slices)",
    opacity=0.8,
)
fig_matrix.update_traces(diagonal_visible=False, showupperhalf=False, marker=dict(size=4))
fig_matrix.update_layout(height=700)
fig_matrix.show()


## §2 6 panel 4D pair plot


In [ ]:
from itertools import combinations

import plotly.graph_objects as go
from plotly.subplots import make_subplots

names = ["length", "weight", "fin", "age"]
pairs = list(combinations(range(4), 2))
fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[f"{names[i]} vs {names[j]}" for i, j in pairs],
)

for ax_idx, (i, j) in enumerate(pairs, start=1):
    row = (ax_idx - 1) // 3 + 1
    col = (ax_idx - 1) % 3 + 1
    for species, color in [("domi", "orange"), ("bream", "steelblue")]:
        sub = df4[df4["species"] == species]
        fig.add_trace(
            go.Scatter(
                x=sub[names[i]],
                y=sub[names[j]],
                mode="markers",
                name=species,
                marker=dict(size=6, color=color, opacity=0.75, line=dict(width=0.3, color="black")),
                showlegend=(ax_idx == 1),
            ),
            row=row,
            col=col,
        )

fig.update_layout(title="4 features: 6 pair-wise 2D slices", height=550)
fig.show()


## §3 parallel coordinates (5D)


In [ ]:
import plotly.express as px

fig_par = px.parallel_coordinates(
    df5,
    dimensions=["length", "weight", "fin", "age", "brightness"],
    color=df5["species"].map({"domi": 0, "bream": 1}),
    color_continuous_scale=[[0, "orange"], [1, "steelblue"]],
    labels={"color": "species code"},
    title="5 features: parallel coordinates (each line = one fish)",
)
fig_par.update_layout(height=450)
fig_par.show()


## §4 k-NN in 4D and 5D

`NEW_FISH_*` 좌표를 바꿔 예측을 확인합니다.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

NEW_FISH_4D = [25, 150, 8.5, 2.5]
NEW_FISH_5D = [25, 150, 8.5, 2.5, 0.45]
K = 3

for name, cols, new_point in [
    ("4D", ["length", "weight", "fin", "age"], NEW_FISH_4D),
    ("5D", ["length", "weight", "fin", "age", "brightness"], NEW_FISH_5D),
]:
    X = df5[cols].values
    y = df5["species"].values
    model = KNeighborsClassifier(n_neighbors=K)
    model.fit(X, y)
    pred = model.predict([new_point])[0]
    dists, idxs = model.kneighbors([new_point])
    print(f"=== {name} k-NN (k={K}) ===")
    print(f"NEW_FISH = {new_point} -> {pred}")
    for rank, (idx, dist) in enumerate(zip(idxs[0], dists[0]), 1):
        row = df5.iloc[idx]
        print(f'  {rank}. {row["species"]:5s} dist={dist:.3f}')
    print()
